# Stage 02 · Step 1 — PatchTST self-supervised pretraining

Masked-patch reconstruction on train printers' multivariate telemetry. The encoder learns rich representations of (weather, workload, health, life, λ) trajectories without any RUL labels.

Outputs:
- `models/ssl_encoder.pt` — encoder state_dict.
- `models/ssl_config.json` — `PatchTSTConfig` so the head can rebuild the same backbone.
- `models/feature_scaler.npz` — per-channel mean / std fit on train printers only.

Hyperparameters are exposed at the top of cell 2; an Optuna HPO sweep is provided in the optional last cell.

In [1]:
from __future__ import annotations
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import PatchTSTConfig, PatchTSTForPretraining

from ml import PROJECT_ROOT
from ml.lib.data import DEFAULT_FLEET_PATH, TRAIN_PRINTERS, filter_printers, load_fleet
from ml.lib.features import base_feature_columns, build_feature_matrix

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
POLICY_DIR = PROJECT_ROOT / 'data/policy_runs'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| GPUs:', torch.cuda.device_count())

from ml.lib.fast import PRETRAIN_EPOCHS, banner
banner()


Device: cuda | GPUs: 2
[fast] mode=FULL · parallel=12 · trials=200/500 · K=60 · epochs=20/3 · ppo_ts=2000/20000 · seeds=(0, 1, 2)


In [2]:
@dataclass
class TrainCfg:
    context_length: int = 360       # ~1 year of daily telemetry per window
    patch_length: int = 30           # ~1 month per patch
    patch_stride: int = 30
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 4
    dropout: float = 0.2
    mask_ratio: float = 0.4
    batch_size: int = 64
    lr: float = 3e-4
    weight_decay: float = 1e-4
    epochs: int = PRETRAIN_EPOCHS  # was 20; toggled by FAST_MODE
    grad_clip: float = 1.0
    seed: int = 0

cfg = TrainCfg()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
cfg

TrainCfg(context_length=360, patch_length=30, patch_stride=30, d_model=256, n_heads=8, n_layers=4, dropout=0.2, mask_ratio=0.4, batch_size=64, lr=0.0003, weight_decay=0.0001, epochs=20, grad_clip=1.0, seed=0)

In [3]:
def load_train_corpus() -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    baseline = filter_printers(load_fleet(DEFAULT_FLEET_PATH), TRAIN_PRINTERS)
    baseline['policy_id'] = -1
    frames.append(baseline)
    if POLICY_DIR.exists():
        for k, path in enumerate(sorted(POLICY_DIR.glob('policy_*.parquet'))):
            extra = filter_printers(pd.read_parquet(path), TRAIN_PRINTERS)
            extra['policy_id'] = k
            frames.append(extra)
    return pd.concat(frames, ignore_index=True)

train_df = load_train_corpus()
feat_df, feature_cols = build_feature_matrix(train_df)
print('Rows:', len(feat_df), '| Features:', len(feature_cols))
feat_df.head(3)

Rows: 15598310 | Features: 44


,printer_id,city,climate_zone,date,day,ambient_temp_c,humidity_pct,dust_concentration,Q_demand,daily_print_hours,...,rul_C3,rul_C4,rul_C5,rul_C6,rul_system,policy_id,sin_doy,cos_doy,sin_month,cos_month
0,0,singapore,tropical,2016-01-01,0,22.580000,54.799999,50.000000,1.0,6.425403,...,41,206,496,2278,23,-1,0.017202,0.999852,0.5,0.866025
1,0,singapore,tropical,2016-01-02,1,22.580000,55.150002,50.000263,1.0,1.017292,...,40,205,495,2277,22,-1,0.034398,0.999408,0.5,0.866025
2,0,singapore,tropical,2016-01-03,2,22.610001,54.450001,50.001328,1.0,3.441179,...,39,204,494,2276,21,-1,0.051584,0.998669,0.5,0.866025


In [4]:
feature_matrix = feat_df[feature_cols].to_numpy(dtype=np.float32)
channel_mean = feature_matrix.mean(axis=0)
channel_std = feature_matrix.std(axis=0)
channel_std = np.where(channel_std < 1e-6, 1.0, channel_std)
np.savez(MODELS_DIR / 'feature_scaler.npz', mean=channel_mean, std=channel_std, columns=np.array(feature_cols))
print('Saved scaler with', len(feature_cols), 'channels')

Saved scaler with 44 channels


In [5]:
class TrajectoryWindowDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols: list[str], context_length: int,
                 mean: np.ndarray, std: np.ndarray):
        self.context_length = int(context_length)
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        sorted_df = df.sort_values(['policy_id', 'printer_id', 'day']).reset_index(drop=True)
        self.windows: list[np.ndarray] = []
        for (_, _), group in sorted_df.groupby(['policy_id', 'printer_id'], sort=False):
            arr = group[feature_cols].to_numpy(dtype=np.float32)
            T = arr.shape[0]
            if T < self.context_length:
                continue
            for start in range(0, T - self.context_length + 1, self.context_length):
                window = arr[start:start + self.context_length]
                self.windows.append(window)

    def __len__(self) -> int:
        return len(self.windows)

    def __getitem__(self, idx: int) -> torch.Tensor:
        window = self.windows[idx]
        normed = (window - self.mean) / self.std
        return torch.from_numpy(normed)

dataset = TrajectoryWindowDataset(
    feat_df,
    feature_cols,
    cfg.context_length,
    channel_mean,
    channel_std,
)
print('Windows:', len(dataset))
loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True, drop_last=True)

Windows: 42700


In [6]:
patch_cfg = PatchTSTConfig(
    num_input_channels=len(feature_cols),
    context_length=cfg.context_length,
    patch_length=cfg.patch_length,
    patch_stride=cfg.patch_stride,
    d_model=cfg.d_model,
    num_attention_heads=cfg.n_heads,
    num_hidden_layers=cfg.n_layers,
    dropout=cfg.dropout,
    random_mask_ratio=cfg.mask_ratio,
    use_cls_token=False,
)
model = PatchTSTForPretraining(patch_cfg).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {n_params / 1e6:.2f}M')

Trainable params: 2.12M


In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs * max(1, len(loader)))
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
history: list[dict] = []
for epoch in range(cfg.epochs):
    model.train()
    epoch_loss = 0.0
    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=device.type == 'cuda'):
            output = model(past_values=batch)
            loss = output.loss.mean() if output.loss.ndim > 0 else output.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        epoch_loss += float(loss.detach().item())
    avg = epoch_loss / max(1, len(loader))
    history.append({'epoch': epoch, 'loss': avg})
    print(f'epoch {epoch:02d} | loss {avg:.4f}')
history_df = pd.DataFrame(history)
history_df.to_csv(MODELS_DIR / 'pretrain_history.csv', index=False)
history_df.tail()

/home/sterry/Desktop/projects/hackupc2026/.venv/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


epoch 00 | loss 0.2888


epoch 01 | loss 0.1907


epoch 02 | loss 0.1758


epoch 03 | loss 0.1666


...

epoch 16 | loss 0.1208


epoch 17 | loss 0.1199


epoch 18 | loss 0.1191


epoch 19 | loss 0.1190


,epoch,loss
15,15,0.122246
16,16,0.120813
17,17,0.119868
18,18,0.119117
19,19,0.119033


In [8]:
core = model.module if isinstance(model, nn.DataParallel) else model
torch.save(core.state_dict(), MODELS_DIR / 'ssl_encoder.pt')
with (MODELS_DIR / 'ssl_config.json').open('w', encoding='utf-8') as handle:
    json.dump({'patch_cfg': patch_cfg.to_dict(), 'train_cfg': asdict(cfg)}, handle, indent=2)
print('Saved encoder + config to', MODELS_DIR)

Saved encoder + config to /home/sterry/Desktop/projects/hackupc2026/ml_models/02_ssl/models


## Optional — Optuna HPO sweep over the SSL hyperparameters

Skip the cell below for a single-config run; uncomment to launch an Optuna study that jointly tunes (`d_model`, `n_layers`, `mask_ratio`, `lr`, `weight_decay`, `patch_length`). Each trial trains for a few epochs and reports validation reconstruction loss on a held-out slice of the train printers.

In [9]:
# import optuna
# def hpo_objective(trial: optuna.Trial) -> float:
#     trial_cfg = TrainCfg(
#         d_model=trial.suggest_categorical('d_model', [128, 256, 512]),
#         n_layers=trial.suggest_int('n_layers', 3, 6),
#         mask_ratio=trial.suggest_float('mask_ratio', 0.3, 0.5),
#         lr=trial.suggest_float('lr', 1e-5, 1e-3, log=True),
#         weight_decay=trial.suggest_float('wd', 1e-6, 1e-2, log=True),
#         patch_length=trial.suggest_categorical('patch_length', [15, 30, 60]),
#         epochs=4,  # short budget per trial
#     )
#     ...  # rebuild dataset / model with trial_cfg, train, return val loss
# study = optuna.create_study(direction='minimize',
#                             sampler=optuna.samplers.TPESampler(seed=42),
#                             pruner=optuna.pruners.MedianPruner())
# study.optimize(hpo_objective, n_trials=20)
# print(study.best_params)